### Lab 7 Meeting 2 ###

In [7]:
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, state_fidelity
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_bloch_multivector, plot_state_qsphere, plot_histogram
import matplotlib.pyplot as plt
import numpy as np

sim = AerSimulator()

In [8]:
def run_full_adder(A,B,Cin):
    qc = QuantumCircuit(5,2)

    if A: qc.x(0)
    if B: qc.x(1)
    if Cin: qc.x(2)

    qc.cx(0,3)
    qc.cx(1,3)
    qc.cx(2,3)

    qc.ccx(0,1,4)
    qc.ccx(0,2,4)
    qc.ccx(1,2,4)

    qc.measure(3,0)
    qc.measure(4,1)

    compiled = transpile(qc, sim)
    result = sim.run(compiled, shots = 1).result()

    bits = list(result.get_counts().keys())[0]
    return int(bits[1]), int(bits[0])

In [123]:
print("A","B","C", f"->", "(S,", "O)")
for A in [0,1]:
    for B in [0,1]:
        for Cin in [0,1]:
            print(A,B,Cin, f"->", run_full_adder(A,B,Cin))

A B C -> (S, O)
0 0 0 -> (0, 0)
0 0 1 -> (1, 0)
0 1 0 -> (1, 0)
0 1 1 -> (0, 1)
1 0 0 -> (1, 0)
1 0 1 -> (0, 1)
1 1 0 -> (0, 1)
1 1 1 -> (1, 1)


In [ ]:
def run_full_adder_n_bit(A,B,Cin, numOfBits): 
    qc = QuantumCircuit(4*numOfBits+1,numOfBits+1)
    A.reverse()
    B.reverse()

    # Initialize qubits
    A_bits = []
    B_bits = []
    C_bits = []
    Sum_bits = []
    for i in range(0,numOfBits):
        A_bits.append(i)
        B_bits.append(i + numOfBits)
        C_bits.append(i + (2*numOfBits))
        Sum_bits.append(i + 1 +(3*numOfBits))
    C_bits.append(3*numOfBits)

    # Initialize qubit inputs
    for i in range(0,numOfBits):
        if A[i]: qc.x(A_bits[i])
        if B[i]: qc.x(B_bits[i])

    if Cin: qc.x(C_bits[0])


    for i in range(0,numOfBits):
        # Add A,B and Cin 
        qc.cx(A_bits[i],Sum_bits[i])
        qc.cx(B_bits[i],Sum_bits[i])
        qc.cx(C_bits[i],Sum_bits[i])

        # Compute the carry bit
        qc.ccx(A_bits[i],B_bits[i],C_bits[i+1])
        qc.ccx(A_bits[i],C_bits[i],C_bits[i+1])
        qc.ccx(B_bits[i],C_bits[i],C_bits[i+1])

    # Measure the sums and last carry
    for i in range(0,numOfBits):
        qc.measure(Sum_bits[i],i)
    qc.measure(C_bits[numOfBits],numOfBits)

    compiled = transpile(qc, sim)
    result = sim.run(compiled, shots = 1).result()

    bits = list(result.get_counts().keys())[0]
    A.reverse()
    B.reverse()
    cout = bits[0]
    sum = bits[1:]
    return sum,cout, qc


In [ ]:
def full_adder_sum_checker(n):
    print(f"{"A"}", "+", f"{"B"}", "+", f"{"C"}", "=", f"Out")
    for cin in [0,1]:
        for a in range(0, 2**n):
            a_bits = [int(a_bit) for a_bit in format(a, f"0{n}b")]
            for b in range(0, 2**n):
                b_bits = [int(b_bit) for b_bit in format(b, f"0{n}b")]
                sum, cout, _ = run_full_adder_n_bit(a_bits,b_bits,int(cin), int(n))
                
                a_string = format(a, f"0{n}b")
                b_string = format(b, f"0{n}b")
                print(f"{a}", "+ " f"{b}", "+", f"{cin}", "= " f"{int(sum + cout,2)}")

In [ ]:
def full_adder_truth_table_generator(n):
    # Generate every combination of A, B, and Cin
    # print(f"{"A":^{n}}", f"{"B":^{n}}", f"{"C"}", f"{"S":^{n}}",f"{"C"}")
    for cin in [0,1]:
        for a in range(0, 2**n):
            a_bits = [int(a_bit) for a_bit in format(a, f"0{n}b")]
            for b in range(0, 2**n):
                b_bits = [int(b_bit) for b_bit in format(b, f"0{n}b")]
                sum, cout,_ = run_full_adder_n_bit(a_bits,b_bits,int(cin), int(n))
                a_string = format(a, f"0{n}b")
                b_string = format(b, f"0{n}b")
                # print(f"{a_string}", f"{b_string}", f"{cin}", f"{sum}", f"{cout}")
                print(f"{a_string}", f"{"&"}", f"{b_string}", f"{"&"}", f"{cin}", f"{"&"}", f"{sum}", f"{"&"}", f"{cout}", f"{"&"}", f"{a+b} = {int(cout+sum,2)}", f"{"\\\\ \\hline"}")


In [ ]:
qc = run_full_adder_n_bit([0,1,1,0],[1,0,0,1],0, 4)[2]

In [ ]:
qc = run_full_adder_n_bit([0],[1],0, 1)[2]

In [23]:
qc.count_ops()

OrderedDict([('cx', 3), ('ccx', 3), ('measure', 2), ('x', 1)])

In [24]:
qc.depth()

6

In [359]:
full_adder_sum_checker(3)

A + B + C = Out
0 + 0 + 0 = 0
0 + 1 + 0 = 1
0 + 2 + 0 = 2
0 + 3 + 0 = 3
0 + 4 + 0 = 4
0 + 5 + 0 = 5
0 + 6 + 0 = 6
0 + 7 + 0 = 7
1 + 0 + 0 = 1
1 + 1 + 0 = 2
1 + 2 + 0 = 3
1 + 3 + 0 = 4
1 + 4 + 0 = 5
1 + 5 + 0 = 6
1 + 6 + 0 = 7
1 + 7 + 0 = 8
2 + 0 + 0 = 2
2 + 1 + 0 = 3
2 + 2 + 0 = 4
2 + 3 + 0 = 5
2 + 4 + 0 = 6
2 + 5 + 0 = 7
2 + 6 + 0 = 8
2 + 7 + 0 = 9
3 + 0 + 0 = 3
3 + 1 + 0 = 4
3 + 2 + 0 = 5
3 + 3 + 0 = 6
3 + 4 + 0 = 7
3 + 5 + 0 = 8
3 + 6 + 0 = 9
3 + 7 + 0 = 10
4 + 0 + 0 = 4
4 + 1 + 0 = 5
4 + 2 + 0 = 6
4 + 3 + 0 = 7
4 + 4 + 0 = 8
4 + 5 + 0 = 9
4 + 6 + 0 = 10
4 + 7 + 0 = 11
5 + 0 + 0 = 5
5 + 1 + 0 = 6
5 + 2 + 0 = 7
5 + 3 + 0 = 8
5 + 4 + 0 = 9
5 + 5 + 0 = 10
5 + 6 + 0 = 11
5 + 7 + 0 = 12
6 + 0 + 0 = 6
6 + 1 + 0 = 7
6 + 2 + 0 = 8
6 + 3 + 0 = 9
6 + 4 + 0 = 10
6 + 5 + 0 = 11
6 + 6 + 0 = 12
6 + 7 + 0 = 13
7 + 0 + 0 = 7
7 + 1 + 0 = 8
7 + 2 + 0 = 9
7 + 3 + 0 = 10
7 + 4 + 0 = 11
7 + 5 + 0 = 12
7 + 6 + 0 = 13
7 + 7 + 0 = 14
0 + 0 + 1 = 1
0 + 1 + 1 = 2
0 + 2 + 1 = 3
0 + 3 + 1 = 4
0 + 4 + 1 = 5
0 +

In [18]:
full_adder_truth_table_generator(4)

0000 & 0000 & 0 & 0000 & 0 & 0 = 0 \\ \hline
0000 & 0001 & 0 & 0001 & 0 & 1 = 1 \\ \hline
0000 & 0010 & 0 & 0010 & 0 & 2 = 2 \\ \hline
0000 & 0011 & 0 & 0011 & 0 & 3 = 3 \\ \hline
0000 & 0100 & 0 & 0100 & 0 & 4 = 4 \\ \hline
0000 & 0101 & 0 & 0101 & 0 & 5 = 5 \\ \hline
0000 & 0110 & 0 & 0110 & 0 & 6 = 6 \\ \hline
0000 & 0111 & 0 & 0111 & 0 & 7 = 7 \\ \hline
0000 & 1000 & 0 & 1000 & 0 & 8 = 8 \\ \hline
0000 & 1001 & 0 & 1001 & 0 & 9 = 9 \\ \hline
0000 & 1010 & 0 & 1010 & 0 & 10 = 10 \\ \hline
0000 & 1011 & 0 & 1011 & 0 & 11 = 11 \\ \hline
0000 & 1100 & 0 & 1100 & 0 & 12 = 12 \\ \hline
0000 & 1101 & 0 & 1101 & 0 & 13 = 13 \\ \hline
0000 & 1110 & 0 & 1110 & 0 & 14 = 14 \\ \hline
0000 & 1111 & 0 & 1111 & 0 & 15 = 15 \\ \hline
0001 & 0000 & 0 & 0001 & 0 & 1 = 1 \\ \hline
0001 & 0001 & 0 & 0010 & 0 & 2 = 2 \\ \hline
0001 & 0010 & 0 & 0011 & 0 & 3 = 3 \\ \hline
0001 & 0011 & 0 & 0100 & 0 & 4 = 4 \\ \hline
0001 & 0100 & 0 & 0101 & 0 & 5 = 5 \\ \hline
0001 & 0101 & 0 & 0110 & 0 & 6 = 6 \\ \hlin

In [166]:
string_chars = "01101"
[int(char) for char in string_chars]

[0, 1, 1, 0, 1]

In [236]:
numOfBits = 4
A = [1,0,0,1]
B = [1,1,1,0]
Cin = 0
run_full_adder_2_bit(A,B,Cin, numOfBits)

('1',
 '0111',
 '10111')